In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
RAW_DIR = PROJECT_DIR / "data" / "raw"

# Le meme inventaire est reutilise dans les notebooks suivants.
SOURCE_PATTERNS = {
    "Moov": "moov.csv",
    "Togocom": "Togocom.csv",
    "Telecom": "file-Agences*.csv",
    "CANAL+": "canalplus.csv",
    "Data center": "datacenter.csv",
    "Mobile Money": "mobile money.csv",
}


def find_source(pattern):
    matches = list(RAW_DIR.glob(pattern))
    return matches[0] if matches else None


def read_csv_safe(path):
    for encoding in ("utf-8-sig", "latin1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Encodage impossible : {path.name}")


print("CHARGEMENT DES DONNEES")
source_summary = []
dfs = {}
for service, pattern in SOURCE_PATTERNS.items():
    path = find_source(pattern)
    if path is None:
        print(f"Fichier absent : {pattern}")
        continue
    frame = read_csv_safe(path)
    dfs[service] = frame
    source_summary.append({"service": service, "fichier": path.name, "lignes": len(frame), "colonnes": len(frame.columns)})

catalogue = pd.DataFrame(source_summary)
display(catalogue)

print("\nAPERÇU DES COLONNES")
for service, frame in dfs.items():
    print(f"{service}: {frame.columns.tolist()}")
    display(frame.head(3))

GEOJSON_PATTERNS = {
    "Régions": "Limites administratives - Régions.json",
    "Préfectures": "Limites administratives - Préféctures.json",
    "Communes": "Limites administratives - Communes.json",
    "Cantons": "Limites administratives - Cantons.json",
}
gdfs = {}
for level, filename in GEOJSON_PATTERNS.items():
    path = RAW_DIR / filename
    if path.exists():
        gdfs[level] = gpd.read_file(path)
        print(f"{level}: {len(gdfs[level])} géométries, CRS={gdfs[level].crs}")

# Objets contractuels transmis au notebook 02 : dfs, gdfs, RAW_DIR et PROJECT_DIR.